# Übung zu Vorlesung 6: Ein CDSS selbst bauen – und prüfen

**Informationssysteme im Gesundheitswesen · WiSe 2026/27**

In der Vorlesung haben wir die CDSS-Kette kennengelernt:

**DATA → TRIGGER → LOGIC → OUTPUT → ACTION → FEEDBACK**

Heute bauen Sie den mittleren Teil selbst: eine **Regel**, ein **Modell**, einen **Threshold** – und Sie prüfen, ob das Ganze
in einer zweiten Klinik noch funktioniert.

| Teil | Thema | ca. Zeit |
|---|---|---:|
| 0 | Setup | 5 min |
| 1 | Daten kennenlernen | 10 min |
| 2 | Eine Regel als Funktion – mit Tests | 15 min |
| 3 | Ein Modell trainieren | 10 min |
| 4 | Den Threshold wählen | 15 min |
| 5 | Eine verdächtig gute Variable | 8 min |
| 6 | Externe Validierung: Klinik B | 15 min |
| 7 | Was bedeutet ein fehlender Wert? | 7 min |
| 8 | Vom Modell zum CDSS | 5 min |

> ⚠️ **Wichtig:** Der Datensatz ist **vollständig synthetisch**. Das Outcome `verschlechterung_6h` ist eine
> didaktisch definierte „klinische Verschlechterung innerhalb von 6 Stunden“ – **keine Sepsisdiagnose**.
> Was Sie heute bauen, ist ein **Lehrprototyp**, kein klinisch einsetzbares System.

## Teil 0 – Setup (5 min)

Führen Sie die folgende Zelle aus. Sie lädt die Bibliotheken und definiert eine Hilfsfunktion `kennzahlen()`,
die wir immer wieder brauchen.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, roc_auc_score
from sklearn.calibration import calibration_curve

pd.set_option("display.precision", 2)


def kennzahlen(y_true, y_alert):
    """Berechnet die wichtigsten Kennzahlen eines Alerts (0/1) gegenüber dem tatsächlichen Outcome (0/1)."""
    tn, fp, fn, tp = confusion_matrix(y_true, y_alert, labels=[0, 1]).ravel()
    return {
        "TP": tp, "FP": fp, "FN": fn, "TN": tn,
        "Sensitivität": tp / (tp + fn) if tp + fn else np.nan,
        "Spezifität": tn / (tn + fp) if tn + fp else np.nan,
        "PPV": tp / (tp + fp) if tp + fp else np.nan,
        "Alerts pro 100 Pat.": 100 * (tp + fp) / len(y_true),
    }

print("Setup ok")

## Teil 1 – Daten kennenlernen (10 min)

Zwei Kliniken, gleiche Variablen. Jede Zeile ist eine Patientin bzw. ein Patient zum **Vorhersagezeitpunkt**.

In [ ]:
a = pd.read_csv("klinik_A.csv")
b = pd.read_csv("klinik_B.csv")
print("Klinik A:", a.shape, " Klinik B:", b.shape)
a.head()

In [ ]:
# Aufgabe 1a: Wie hoch ist die Ereignisrate (Anteil verschlechterung_6h == 1) in Klinik A und in Klinik B?
# Tipp: .mean() auf einer 0/1-Spalte ergibt den Anteil.
rate_a = None  # TODO
rate_b = None  # TODO
assert rate_a is not None and rate_b is not None, "Bitte zuerst rate_a und rate_b berechnen."
print(f"Ereignisrate A: {rate_a:.1%}   B: {rate_b:.1%}")

In [ ]:
# Ein Blick auf die Verteilungen
a.describe().T[["mean", "std", "min", "max"]]

**Aufgabe 1b (Diskussion):** Gehen Sie die Spalten durch. Welche Information wäre zum Vorhersagezeitpunkt
**tatsächlich verfügbar** – und bei welcher sind Sie unsicher? Notieren Sie Ihre Vermutung, wir kommen in Teil 5 darauf zurück.

*Ihre Antwort:* …

## Teil 2 – Eine Regel als Funktion, mit Tests (15 min)

Wir implementieren eine **qSOFA-artige Regel** (didaktisch vereinfacht):

- Atemfrequenz ≥ 22/min → 1 Punkt
- systolischer Blutdruck ≤ 100 mmHg → 1 Punkt
- veränderte Bewusstseinslage → 1 Punkt
- **Alert, wenn mindestens 2 Punkte**

**Aufgabe 2a:** Vervollständigen Sie die Funktion.

In [ ]:
def qsofa_alert(atemfrequenz, rr_systolisch, bewusstsein_veraendert):
    """Gibt True zurück, wenn mindestens 2 der 3 Kriterien erfüllt sind."""
    punkte = 0
    # TODO: Punkte vergeben
    ...
    return punkte >= 2

**Aufgabe 2b – Tests:** Eine Regel ist Software – und Software wird getestet. Die ersten beiden Tests sind vorgegeben
(Maria K. aus der Vorlesung und ein unauffälliger Patient). Ergänzen Sie **zwei eigene Tests für die Grenzwerte**
(was passiert genau bei Atemfrequenz 22 und RR 100?).

In [ ]:
# Referenzfälle
assert qsofa_alert(atemfrequenz=26, rr_systolisch=96, bewusstsein_veraendert=0) == True    # Maria K.
assert qsofa_alert(atemfrequenz=18, rr_systolisch=125, bewusstsein_veraendert=0) == False  # unauffällig

# TODO: zwei Grenzwert-Tests
# assert qsofa_alert(...) == ...
# assert qsofa_alert(...) == ...
print("Alle Tests bestanden")

**Aufgabe 2c – Regressionstest:** Eine Kollegin schlägt vor, die Atemfrequenz-Schwelle auf `> 22` zu ändern.
Ändern Sie die Funktion testweise und führen Sie die Testzelle erneut aus. Welcher Test schlägt fehl – und was lernen Sie
daraus? (Danach bitte wieder auf `>= 22` zurücksetzen.)

*Ihre Antwort:* …

**Aufgabe 2d:** Wenden Sie die Regel auf alle Patient:innen in Klinik A an und berechnen Sie die Kennzahlen.

In [ ]:
a["alert_regel"] = a.apply(
    lambda r: qsofa_alert(r["atemfrequenz"], r["rr_systolisch"], r["bewusstsein_veraendert"]), axis=1)

# TODO: Kennzahlen der Regel in Klinik A
# kennzahlen(...)

**Frage:** Die Spezifität ist hoch, die Sensitivität niedrig. Passt das zur Aussage aus der Vorlesung, dass qSOFA
**nicht als alleiniges Screening-Instrument** empfohlen wird?

*Ihre Antwort:* …

## Teil 3 – Ein Modell trainieren (10 min)

Jetzt lernen wir die Entscheidungsfunktion aus Daten: eine **logistische Regression** auf Klinik A
(70 % Training, 30 % Test).

In [ ]:
MERKMALE = ["alter", "herzfrequenz", "atemfrequenz", "rr_systolisch", "temperatur", "spo2",
            "kreatinin", "leukozyten", "crp", "bewusstsein_veraendert"]
ZIEL = "verschlechterung_6h"

train_a, test_a = train_test_split(a, test_size=0.3, random_state=42, stratify=a[ZIEL])

modell = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
modell.fit(train_a[MERKMALE], train_a[ZIEL])

# Das Modell gibt Wahrscheinlichkeiten aus, keine Ja/Nein-Entscheidung:
p_test_a = modell.predict_proba(test_a[MERKMALE])[:, 1]
p_test_a[:5]

In [ ]:
# Aufgabe 3a: Berechnen Sie die AUC des Modells auf den Testdaten von Klinik A.
auc_a = None  # TODO
assert auc_a is not None, "Bitte zuerst auc_a berechnen."
print(f"AUC Klinik A (Test): {auc_a:.3f}")

**Frage:** Die AUC sagt, wie gut das Modell Patient:innen mit und ohne Ereignis *ordnet*. Kann man mit einer
AUC allein schon einen Alert auslösen? Was fehlt noch?

*Ihre Antwort:* …

## Teil 4 – Den Threshold wählen (15 min)

**Aufgabe 4a:** Berechnen Sie die Kennzahlen für die Thresholds 0,1 bis 0,8.

In [ ]:
zeilen = []
for t in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    # TODO: Alert = Wahrscheinlichkeit >= t; Kennzahlen berechnen und mit dem Threshold an 'zeilen' anhängen
    # Format jeder Zeile: ein Dictionary mit dem Schlüssel "Threshold" plus den Kennzahlen,
    # z. B. zeilen.append({"Threshold": t, **kennzahlen(..., ...)})
    ...
assert zeilen, "Bitte zuerst die Schleife vervollständigen (siehe Format oben)."
tabelle = pd.DataFrame(zeilen).set_index("Threshold")
tabelle

In [ ]:
fig, ax1 = plt.subplots(figsize=(7, 4))
ax1.plot(tabelle.index, tabelle["Sensitivität"], marker="o", label="Sensitivität")
ax1.plot(tabelle.index, tabelle["PPV"], marker="o", label="PPV")
ax1.set_xlabel("Threshold"); ax1.set_ylabel("Anteil"); ax1.set_ylim(0, 1)
ax2 = ax1.twinx()
ax2.bar(tabelle.index, tabelle["Alerts pro 100 Pat."], width=0.04, alpha=0.25, color="grey", label="Alerts/100")
ax2.set_ylabel("Alerts pro 100 Patient:innen")
ax1.legend(loc="upper center"); plt.title("Klinik A: Was der Threshold verändert"); plt.show()

**Aufgabe 4b – Ihre Entscheidung:** Die Station sagt: *„Mehr als 15 Alerts pro 100 Patient:innen schauen wir
uns nicht mehr ernsthaft an.“* Welchen Threshold wählen Sie? Wie hoch sind dann Sensitivität und PPV?
Und: Wer sollte diese Entscheidung eigentlich treffen?

*Ihre Antwort:* …

In [ ]:
# Legen Sie Ihren gewählten Threshold fest (wird in Teil 6 wiederverwendet)
THRESHOLD = None  # hier Ihre Entscheidung aus Aufgabe 4b eintragen
assert THRESHOLD is not None, "Bitte zuerst Ihren Threshold aus Aufgabe 4b eintragen."
print("THRESHOLD =", THRESHOLD)

## Teil 5 – Eine verdächtig gute Variable (8 min)

**Aufgabe 5a:** Trainieren Sie dasselbe Modell noch einmal, diesmal **zusätzlich** mit `antibiotikum_gegeben`.
Wie verändert sich die AUC?

In [ ]:
MERKMALE_PLUS = MERKMALE + ["antibiotikum_gegeben"]
# TODO: Modell 'modell_leak' trainieren und AUC auf test_a berechnen


**Aufgabe 5b:** Das Modell wird deutlich „besser“. Würden Sie es so ausrollen? Beantworten Sie die Frage:
**Wann war diese Information verfügbar?**

*Ihre Antwort:* …

## Teil 6 – Externe Validierung: Klinik B (15 min)

Ihr Modell (ohne Leakage-Variable) wird in Klinik B eingesetzt. Dort sind die Patient:innen älter, und
Verschlechterungen sind seltener.

**Aufgabe 6a:** Berechnen Sie die AUC und die Kennzahlen bei Ihrem `THRESHOLD` für Klinik B und vergleichen Sie mit Klinik A.

In [ ]:
p_b = modell.predict_proba(b[MERKMALE])[:, 1]
# TODO: AUC für Klinik B; Kennzahlen bei THRESHOLD für A (Test) und B nebeneinander


**Aufgabe 6b – Kalibrierung:** Die AUC ist fast gleich. Aber stimmen die vorhergesagten Risiken?
Vergleichen Sie mittlere Vorhersage und beobachtete Rate, und zeichnen Sie die Kalibrierungskurven.

In [ ]:
print(f"Klinik B – mittleres vorhergesagtes Risiko: {p_b.mean():.1%}   beobachtete Rate: {b[ZIEL].mean():.1%}")

fig, ax = plt.subplots(figsize=(5, 5))
for name, y, p in [("Klinik A (Test)", test_a[ZIEL], p_test_a), ("Klinik B", b[ZIEL], p_b)]:
    beob, vorh = calibration_curve(y, p, n_bins=6, strategy="quantile")
    ax.plot(vorh, beob, marker="o", label=name)
ax.plot([0, 1], [0, 1], "k--", label="perfekt kalibriert")
ax.set_xlabel("vorhergesagtes Risiko"); ax.set_ylabel("beobachtete Ereignisrate")
ax.legend(); plt.title("Kalibrierung"); plt.show()

**Frage:** Was bedeutet es für die Ärztin in Klinik B, wenn das System „Risiko 40 %“ anzeigt?

*Ihre Antwort:* …

In [ ]:
# Bonus: Rekalibrierung des Achsenabschnitts (logit-Verschiebung), z. B. mit einer kleinen Stichprobe aus Klinik B.
# Idee: logit(p_neu) = logit(p_alt) + delta, wobei delta so gewählt wird, dass der Mittelwert zur Rate in B passt.


## Teil 7 – Was bedeutet ein fehlender Wert? (7 min)

`laktat` wurde nicht bei allen Patient:innen gemessen.

**Aufgabe 7a:** Vergleichen Sie den Anteil fehlender Laktatwerte – getrennt nach Outcome – in Klinik A und B.

In [ ]:
# TODO: Anteil fehlender Laktatwerte je Klinik und Outcome
# Tipp: df.groupby(ZIEL)["laktat"].apply(lambda s: s.isna().mean())


> **Hinweis:** Das folgende Modell wird auf den Daten von Klinik B trainiert *und* dort interpretiert. Es dient
> **nur zur Demonstration des Zusammenhangs** und ist keine valide Modell-Evaluation. Uns interessiert allein der
> Koeffizient von `laktat_fehlt`, nicht die Vorhersageleistung.

In [ ]:
# Ein Modell in Klinik B, das zusätzlich weiß, OB Laktat gemessen wurde
b["laktat_fehlt"] = b["laktat"].isna().astype(int)
modell_b = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
modell_b.fit(b[MERKMALE + ["laktat_fehlt"]], b[ZIEL])
koef = pd.Series(modell_b[-1].coef_[0], index=MERKMALE + ["laktat_fehlt"]).sort_values()
koef

**Aufgabe 7b:** Welches Vorzeichen hat der Koeffizient von `laktat_fehlt` – und was hat das Modell damit
eigentlich gelernt? Wäre dieses Wissen auf Klinik A übertragbar?

*Ihre Antwort:* …

## Teil 8 – Vom Modell zum CDSS (5 min)

Füllen Sie die CDSS-Kette für Ihr System aus – in Stichworten:

| Glied | Ihr System |
|---|---|
| **DATA** – welche Daten, wie aktuell? | … |
| **TRIGGER** – wann läuft es (z. B. neuer Vitalwert, `patient-view`)? | … |
| **LOGIC** – Regel, Modell oder beides? Welcher Threshold? | … |
| **OUTPUT** – Info, Warnung, Hard Stop? An wen? | … |
| **ACTION** – was soll die Person konkret tun? | … |
| **FEEDBACK** – welche Kennzahlen überwachen Sie nach dem Go-live? | … |

**Abschlussfrage:** Ihr Notebook ist ein Lehrprototyp. Was müsste sich ändern, damit es zu einem Medizinprodukt wird –
und welche Rolle spielt dabei die **Zweckbestimmung**?